# 06b - Modélisation ML : Prédiction blocs électoraux 2025–2027 — France métropolitaine

**Objectif** : Prédire le `bloc_dominant` (Gauche / Centre / Droite / ExtremeDroite) par commune pour 2025, 2026 et 2027 sur ~34 800 communes.

**Données** : `gold_france.features_communes` — ~174 000 lignes, ~34 800 communes × 5 années (2002, 2007, 2012, 2017, 2022)

**Approche** :
- Split temporel : train ≤ 2017, test = 2022
- Modèles : Random Forest + GradientBoosting
- Features : CSP, diplômes, démographie, tendances temporelles (lag, évolution), **typologie_territoire**
- Prédictions : extrapolation tendancielle 2025, 2026, 2027
- Résultats → `gold_france.predictions_2025_2027`

## 0. Configuration & imports

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Configuration inline
PG_HOST     = os.environ.get('POSTGRES_HOST',     'postgres')
PG_PORT     = os.environ.get('POSTGRES_PORT',     '5432')
PG_DB       = os.environ.get('POSTGRES_DB',       'mspr813')
PG_USER     = os.environ.get('POSTGRES_USER',     'mspr_user')
PG_PASSWORD = os.environ.get('POSTGRES_PASSWORD', 'mspr_password')

DB_URL = f'postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}'
engine = create_engine(DB_URL, pool_pre_ping=True)

print('PostgreSQL OK :', engine.connect().execute(text('SELECT current_database()')).scalar())

PostgreSQL OK : mspr813


## 1. Chargement des données Gold France

In [2]:
print('Chargement gold_france.features_communes...')
df = pd.read_sql(
    'SELECT * FROM gold_france.features_communes ORDER BY code_commune, annee',
    engine
)

print(f'Shape : {df.shape}')
print(f'Communes : {df["code_commune"].nunique()}')
print(f'Années   : {sorted(df["annee"].unique())}')
print(f'Blocs    : {df["bloc_dominant"].value_counts().to_dict()}')
print(f'\nTypologies :')
print(df['typologie_territoire'].value_counts().to_string())
df.head(3)

Chargement gold_france.features_communes...


Shape : (173925, 24)
Communes : 34800
Années   : [2002, 2007, 2012, 2017, 2022]
Blocs    : {'Droite': 66285, 'Gauche': 57173, 'ExtremeDroite': 36387, 'Centre': 14080}

Typologies :
typologie_territoire
rural         147621
periurbain     21544
urbain          4760


,code_commune,libelle,code_dep,annee,pct_gauche,pct_centre,pct_droite,pct_extremedroite,pct_divers,bloc_dominant,...,pct_bac_plus,pct_sans_diplome,nb_naissances,nb_deces,solde_naturel,typologie_territoire,created_at,mediane_revenu_disp,gini,taux_pauvrete
0,01001,ABERGEMENT CLEMENCIAT,01,2002,28.806,14.286,32.319,24.590,0.0,Droite,...,29.601,15.491,8,4,4,rural,2026-03-04 12:02:24.913963,25820.0,0.242,None
1,01001,ABERGEMENT CLEMENCIAT,01,2007,24.904,19.732,43.487,11.877,0.0,Droite,...,29.601,15.491,8,4,4,rural,2026-03-04 12:02:24.913963,25820.0,0.242,None
2,01001,ABERGEMENT CLEMENCIAT,01,2012,30.862,10.822,33.066,25.251,0.0,Droite,...,29.601,15.491,5,7,-2,rural,2026-03-04 12:02:24.913963,25820.0,0.242,None


## 2. Feature Engineering temporel

Ajout de features dérivées : tendances entre élections, lag, encodage de la typologie.

In [3]:
df = df.sort_values(['code_commune', 'annee']).reset_index(drop=True)

# Lag : résultat de l'élection précédente (par commune)
for col in ['pct_gauche', 'pct_centre', 'pct_droite', 'pct_extremedroite']:
    df[f'{col}_lag1'] = df.groupby('code_commune')[col].shift(1)

# Tendance : évolution par rapport à t-1
for col in ['pct_gauche', 'pct_centre', 'pct_droite', 'pct_extremedroite']:
    df[f'{col}_trend'] = df[col] - df[f'{col}_lag1']

# Bloc dominant à t-1
df['bloc_lag1'] = df.groupby('code_commune')['bloc_dominant'].shift(1)

# Intervalle depuis la dernière élection (années)
df['annee_lag1'] = df.groupby('code_commune')['annee'].shift(1)
df['intervalle'] = df['annee'] - df['annee_lag1']

# Encodage ordinal de la typologie : rural=0, périurbain=1, urbain=2
typo_map = {'rural': 0, 'periurbain': 1, 'urbain': 2}
df['typo_ord'] = df['typologie_territoire'].map(typo_map).fillna(0).astype(int)

print(f'Shape après features temporelles : {df.shape}')
df[['code_commune','annee','pct_gauche','pct_gauche_lag1','pct_gauche_trend','bloc_lag1','typo_ord']].head(10)

Shape après features temporelles : (173925, 36)


,code_commune,annee,pct_gauche,pct_gauche_lag1,pct_gauche_trend,bloc_lag1,typo_ord
0,01001,2002,28.806,NaN,NaN,NaN,0
1,01001,2007,24.904,28.806,-3.902,Droite,0
2,01001,2012,30.862,24.904,5.958,Droite,0
3,01001,2017,19.394,30.862,-11.468,Droite,0
4,01001,2022,21.731,19.394,2.337,Droite,0
5,01002,2002,31.293,NaN,NaN,NaN,0
6,01002,2007,29.825,31.293,-1.468,ExtremeDroite,0
7,01002,2012,37.931,29.825,8.106,Droite,0
8,01002,2017,28.409,37.931,-9.522,Gauche,0
9,01002,2022,38.596,28.409,10.187,Gauche,0


In [4]:
# Définir les features et la target
FEATURES = [
    # Résultats précédents (lag)
    'pct_gauche_lag1', 'pct_centre_lag1', 'pct_droite_lag1', 'pct_extremedroite_lag1',
    # Tendances
    'pct_gauche_trend', 'pct_centre_trend', 'pct_droite_trend', 'pct_extremedroite_trend',
    # CSP
    'cadres_pct', 'ouvriers_pct', 'employes_pct', 'artisans_pct',
    # Diplômes
    'pct_bac_plus', 'pct_sans_diplome',
    # Démographie
    'nb_naissances', 'nb_deces', 'solde_naturel',
    # Territoire
    'typo_ord',
    # Temporel
    'annee',
]

TARGET = 'bloc_dominant'

# Supprimer les lignes sans lag (première élection par commune)
df_ml = df[df['bloc_lag1'].notna()].copy()
print(f'Lignes avec lag disponible : {len(df_ml)} (sur {len(df)} total)')
print(f'Années dans df_ml : {sorted(df_ml["annee"].unique())}')

Lignes avec lag disponible : 139125 (sur 173925 total)


Années dans df_ml : [2007, 2012, 2017, 2022]


## 3. Split temporel : train ≤ 2017 / test = 2022

In [5]:
df_train = df_ml[df_ml['annee'] <= 2017].copy()
df_test  = df_ml[df_ml['annee'] == 2022].copy()

print(f'Train : {len(df_train)} lignes | années : {sorted(df_train["annee"].unique())}')
print(f'Test  : {len(df_test)}  lignes | années : {sorted(df_test["annee"].unique())}')
print(f'\nDistribution target train :')
print(df_train[TARGET].value_counts().to_string())
print(f'\nDistribution target test :')
print(df_test[TARGET].value_counts().to_string())

X_train = df_train[FEATURES]
y_train = df_train[TARGET]
X_test  = df_test[FEATURES]
y_test  = df_test[TARGET]

Train : 104342 lignes | années : [2007, 2012, 2017]
Test  : 34783  lignes | années : [2022]

Distribution target train :
bloc_dominant
Gauche           44755
Droite           41553
ExtremeDroite    13502
Centre            4532

Distribution target test :
bloc_dominant
ExtremeDroite    20193
Centre            9485
Gauche            5079
Droite              26


## 4. Modèle Random Forest

In [6]:
rf_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=3,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

print('Entraînement Random Forest...')
rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)
print(f'Random Forest — Accuracy (test 2022) : {acc_rf:.3f}')
print()
print(classification_report(y_test, y_pred_rf, zero_division=0))

Entraînement Random Forest...


Random Forest — Accuracy (test 2022) : 0.778



               precision    recall  f1-score   support

       Centre       0.87      0.42      0.57      9485
       Droite       0.03      0.77      0.06        26
ExtremeDroite       0.77      0.98      0.86     20193
       Gauche       0.84      0.65      0.73      5079

     accuracy                           0.78     34783
    macro avg       0.63      0.70      0.55     34783
 weighted avg       0.81      0.78      0.76     34783



In [7]:
# Importance des features
feat_imp = pd.Series(
    rf_pipeline.named_steps['model'].feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

print('Top 10 features importances (Random Forest):')
print(feat_imp.head(10).round(4).to_string())

Top 10 features importances (Random Forest):
pct_centre_trend           0.1998
pct_extremedroite_lag1     0.1397
pct_gauche_lag1            0.1321
pct_gauche_trend           0.1308
annee                      0.1095
pct_extremedroite_trend    0.0859
pct_droite_lag1            0.0774
pct_droite_trend           0.0539
pct_centre_lag1            0.0501
pct_bac_plus               0.0051


## 5. Modèle Gradient Boosting

In [8]:
# Encoder les labels pour GBM
# Fit sur TOUS les blocs connus pour éviter les labels inconnus
all_blocs = ['Centre', 'Divers', 'Droite', 'ExtremeDroite', 'Gauche']
le = LabelEncoder()
le.fit(all_blocs)
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)

# Imputer séparément
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

print('Entraînement GradientBoosting...')
gbm = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)
gbm.fit(X_train_imp, y_train_enc)
y_pred_gbm_enc = gbm.predict(X_test_imp)
y_pred_gbm = le.inverse_transform(y_pred_gbm_enc)

acc_gbm = accuracy_score(y_test, y_pred_gbm)
print(f'GradientBoosting — Accuracy (test 2022) : {acc_gbm:.3f}')
print()
print(classification_report(y_test, y_pred_gbm, zero_division=0))

Entraînement GradientBoosting...


GradientBoosting — Accuracy (test 2022) : 0.911



               precision    recall  f1-score   support

       Centre       0.98      0.75      0.85      9485
       Droite       0.63      0.92      0.75        26
ExtremeDroite       0.89      0.99      0.94     20193
       Gauche       0.90      0.89      0.89      5079

     accuracy                           0.91     34783
    macro avg       0.85      0.89      0.86     34783
 weighted avg       0.92      0.91      0.91     34783



In [9]:
# Choisir le meilleur modèle
best_model_name = 'RandomForest' if acc_rf >= acc_gbm else 'GradientBoosting'
best_pipeline   = rf_pipeline if acc_rf >= acc_gbm else None
best_imputer    = imputer
best_model      = rf_pipeline.named_steps['model'] if acc_rf >= acc_gbm else gbm

print(f'Meilleur modèle : {best_model_name} (accuracy={max(acc_rf, acc_gbm):.3f})')

Meilleur modèle : GradientBoosting (accuracy=0.911)


## 6. Prédictions 2025, 2026, 2027

**Stratégie** : Pour chaque commune, on extrapole les tendances observées entre 2017 et 2022.
- `pct_X_lag1` = valeur 2022
- `pct_X_trend` = tendance linéaire entre 2017→2022
- Features structurelles (CSP, diplômes, typologie) = constantes (2022)
- Démographie = extrapolation linéaire


In [10]:
# Construire les features pour 2025, 2026, 2027
df_2022 = df[df['annee'] == 2022].copy()
df_2017 = df[df['annee'] == 2017].copy().set_index('code_commune')

print(f'Communes avec données 2022 : {len(df_2022)}')
print(f'Communes avec données 2017 : {len(df_2017)}')

# Tendance 5 ans (2017→2022) par commune
def get_trend(df_2022_row, col):
    comm = df_2022_row['code_commune']
    val_2022 = df_2022_row.get(col, np.nan)
    if comm in df_2017.index:
        val_2017 = df_2017.loc[comm, col]
        return val_2022 - val_2017
    return 0.0

predictions_rows = []

for target_year in [2025, 2026, 2027]:
    years_ahead = target_year - 2022  # 3, 4, 5
    scale = years_ahead / 5.0  # normaliser par rapport à l'intervalle 5 ans

    for _, row in df_2022.iterrows():
        comm = row['code_commune']

        def extrap(col):
            val = row.get(col, np.nan)
            trend = get_trend(row, col)
            return float(val) + trend * scale if pd.notna(val) else np.nan

        feat_row = {
            # Lag = résultats 2022
            'pct_gauche_lag1':         row.get('pct_gauche', np.nan),
            'pct_centre_lag1':         row.get('pct_centre', np.nan),
            'pct_droite_lag1':         row.get('pct_droite', np.nan),
            'pct_extremedroite_lag1':  row.get('pct_extremedroite', np.nan),
            # Tendance extrapolée
            'pct_gauche_trend':        get_trend(row, 'pct_gauche') * scale,
            'pct_centre_trend':        get_trend(row, 'pct_centre') * scale,
            'pct_droite_trend':        get_trend(row, 'pct_droite') * scale,
            'pct_extremedroite_trend': get_trend(row, 'pct_extremedroite') * scale,
            # CSP (statique)
            'cadres_pct':              row.get('cadres_pct', np.nan),
            'ouvriers_pct':            row.get('ouvriers_pct', np.nan),
            'employes_pct':            row.get('employes_pct', np.nan),
            'artisans_pct':            row.get('artisans_pct', np.nan),
            # Diplômes (statique)
            'pct_bac_plus':            row.get('pct_bac_plus', np.nan),
            'pct_sans_diplome':        row.get('pct_sans_diplome', np.nan),
            # Démographie extrapolée
            'nb_naissances':           extrap('nb_naissances'),
            'nb_deces':                extrap('nb_deces'),
            'solde_naturel':           extrap('solde_naturel'),
            # Territoire (statique)
            'typo_ord':                row.get('typo_ord', 0),
            # Année
            'annee':                   target_year,
            # Clés
            '_code_commune':           comm,
            '_libelle':                row.get('libelle', ''),
            '_code_dep':               row.get('code_dep', ''),
            '_typologie':              row.get('typologie_territoire', ''),
            '_target_year':            target_year,
        }
        predictions_rows.append(feat_row)

df_pred_input = pd.DataFrame(predictions_rows)
print(f'Lignes à prédire : {len(df_pred_input)} ({len(df_2022)} communes × 3 années)')

Communes avec données 2022 : 34783
Communes avec données 2017 : 34791


Lignes à prédire : 104349 (34783 communes × 3 années)


In [11]:
# Prédire avec le meilleur modèle
X_pred = df_pred_input[FEATURES]

if best_model_name == 'RandomForest':
    y_pred_future      = rf_pipeline.predict(X_pred)
    y_pred_future_prob = rf_pipeline.predict_proba(X_pred)
    classes = rf_pipeline.classes_
else:
    X_pred_imp = best_imputer.transform(X_pred)
    y_pred_future_enc  = gbm.predict(X_pred_imp)
    y_pred_future      = le.inverse_transform(y_pred_future_enc)
    y_pred_future_prob = gbm.predict_proba(X_pred_imp)
    classes = le.classes_[gbm.classes_]  # classes réelles apprises par le GBM

# Construire DataFrame résultats
df_results = df_pred_input[['_code_commune','_libelle','_code_dep','_typologie','_target_year']].copy()
df_results.columns = ['code_commune','libelle','code_dep','typologie_territoire','annee']
df_results['bloc_predit'] = y_pred_future

# classes correspond aux colonnes de predict_proba — utiliser les classes du modèle réel
n_classes_proba = y_pred_future_prob.shape[1]
for i, cls in enumerate(classes):
    if i < n_classes_proba:
        df_results[f'prob_{cls.lower()}'] = y_pred_future_prob[:, i].round(3)

print(f'Prédictions générées : {len(df_results)}')
print('\nDistribution prédictions 2027 :')
print(df_results[df_results['annee']==2027]['bloc_predit'].value_counts().to_string())
df_results.head(6)

Prédictions générées : 104349

Distribution prédictions 2027 :
bloc_predit
ExtremeDroite    21280
Centre            9037
Gauche            4456
Droite              10


,code_commune,libelle,code_dep,typologie_territoire,annee,bloc_predit,prob_centre,prob_droite,prob_extremedroite,prob_gauche
0,01001,ABERGEMENT CLEMENCIAT,01,rural,2025,ExtremeDroite,0.026,0.000,0.973,0.001
1,01002,ABERGEMENT DE VAREY,01,rural,2025,Gauche,0.057,0.000,0.000,0.943
2,01004,AMBERIEU EN BUGEY,01,urbain,2025,ExtremeDroite,0.001,0.000,0.657,0.342
3,01005,AMBERIEUX EN DOMBES,01,rural,2025,ExtremeDroite,0.007,0.000,0.992,0.001
4,01006,AMBLEON,01,rural,2025,Centre,0.698,0.006,0.012,0.284
5,01007,AMBRONAY,01,periurbain,2025,ExtremeDroite,0.003,0.000,0.996,0.001


## 7. Sauvegarde des prédictions en base

In [12]:
# Ajouter colonne modèle
df_results['modele'] = best_model_name

# Harmoniser les colonnes prob (certains blocs peuvent ne pas être dans les classes)
for col in ['prob_gauche','prob_centre','prob_droite','prob_extremedroite']:
    if col not in df_results.columns:
        df_results[col] = np.nan

# Vider la table avant insertion
with engine.begin() as conn:
    conn.execute(text('TRUNCATE TABLE gold_france.predictions_2025_2027'))

# Insérer
insert_cols = [
    'code_commune','libelle','code_dep','annee','bloc_predit',
    'prob_gauche','prob_centre','prob_droite','prob_extremedroite',
    'modele','typologie_territoire'
]
df_results[insert_cols].to_sql(
    'predictions_2025_2027',
    engine,
    schema='gold_france',
    if_exists='append',
    index=False,
    method='multi',
    chunksize=1000
)

with engine.connect() as conn:
    n = conn.execute(text('SELECT COUNT(*) FROM gold_france.predictions_2025_2027')).scalar()
print(f'gold_france.predictions_2025_2027 : {n} lignes insérées')

gold_france.predictions_2025_2027 : 104349 lignes insérées


## 8. Résultats finaux

In [13]:
print('=' * 65)
print('RÉSULTATS MODÉLISATION FRANCE MÉTROPOLITAINE')
print('=' * 65)
print(f'  Modèle Random Forest    — accuracy test 2022 : {acc_rf:.3f}')
print(f'  Modèle GradientBoosting — accuracy test 2022 : {acc_gbm:.3f}')
print(f'  Modèle retenu : {best_model_name}')
print()

for yr in [2025, 2026, 2027]:
    sub = df_results[df_results['annee'] == yr]
    dist = sub['bloc_predit'].value_counts()
    print(f'  {yr} — {len(sub)} communes prédites :')
    for bloc, cnt in dist.items():
        print(f'       {bloc:20s} : {cnt:6d} communes ({cnt/len(sub)*100:.1f}%)')
    print()

print('=' * 65)

RÉSULTATS MODÉLISATION FRANCE MÉTROPOLITAINE
  Modèle Random Forest    — accuracy test 2022 : 0.778
  Modèle GradientBoosting — accuracy test 2022 : 0.911
  Modèle retenu : GradientBoosting

  2025 — 34783 communes prédites :
       ExtremeDroite        :  25097 communes (72.2%)
       Centre               :   5067 communes (14.6%)
       Gauche               :   4609 communes (13.3%)
       Droite               :     10 communes (0.0%)

  2026 — 34783 communes prédites :
       ExtremeDroite        :  23480 communes (67.5%)
       Centre               :   6815 communes (19.6%)
       Gauche               :   4479 communes (12.9%)
       Droite               :      9 communes (0.0%)

  2027 — 34783 communes prédites :
       ExtremeDroite        :  21280 communes (61.2%)
       Centre               :   9037 communes (26.0%)
       Gauche               :   4456 communes (12.8%)
       Droite               :     10 communes (0.0%)



In [14]:
# Aperçu prédictions 2027 par typologie
pred_2027 = df_results[df_results['annee'] == 2027].copy()
print('Prédictions 2027 par typologie territoire :')
print(
    pred_2027.groupby(['typologie_territoire', 'bloc_predit'])
    .size()
    .unstack(fill_value=0)
    .to_string()
)

Prédictions 2027 par typologie territoire :
bloc_predit           Centre  Droite  ExtremeDroite  Gauche
typologie_territoire                                       
periurbain               804       0           2950     556
rural                   8146      10          17972    3393
urbain                    87       0            358     507


In [15]:
# Top 10 départements par nombre de communes ExtremeDroite prédites en 2027
pred_xd_2027 = pred_2027[pred_2027['bloc_predit'] == 'ExtremeDroite']
if len(pred_xd_2027) > 0:
    top_dep = pred_xd_2027.groupby('code_dep').size().sort_values(ascending=False).head(10)
    print('Top 10 départements communes ExtremeDroite prédites en 2027 :')
    print(top_dep.to_string())
else:
    print('Aucune commune ExtremeDroite prédite en 2027 (données insuffisantes en train ?)')
    print('Distribution complète 2027 :')
    print(pred_2027['bloc_predit'].value_counts().to_string())

Top 10 départements communes ExtremeDroite prédites en 2027 :
code_dep
02     674
62     593
60     574
57     540
80     522
27     449
76     442
59     431
21     426
33     422
